# Lab 6.1 - AI-Assisted Development with VS Code and Cursor
**Module 6: AI-Powered Developer Tooling & Workflow Automation**

In this lab you will:
- Understand the four core AI-assisted development patterns: **code generation**, **refactoring**, **documentation**, and **unit test generation**
- Use the **Gemini API to simulate** what inline editor AI (Cursor, Copilot, VS Code) does under the hood
- Generate a production-quality `NutanixLogParser` class from a natural-language description
- Refactor messy legacy code into clean, well-structured Python
- Auto-generate **Google-style docstrings** and a project README
- Generate and execute **pytest unit tests** from function signatures

> **Instructor Note:** This lab bridges the prompt engineering skills from Lab 5.2 directly into the editor workflow. Every technique shown here (system prompt, temperature, structured output) is exactly what tools like Cursor and GitHub Copilot send to their backend LLMs. Running it in a notebook first gives engineers the mental model they need to use editor AI deliberately — not as magic, but as a controllable API call.

## Requirements & Troubleshooting

### Required Packages

| Package | Install Name | Purpose |
|---------|-------------|----------|
| google-generativeai | `google-generativeai` | Gemini API client |
| pandas | `pandas` | Data tables and log analysis |

**Install all at once:**
```bash
pip install google-generativeai pandas
```

### API Key Required

This lab calls the **Google Gemini API**. Set your key as an environment variable:

```bash
export GEMINI_API_KEY="AIza..."
```

Or add it to a `.env` file at the project root:
```
GEMINI_API_KEY=AIza...
```

Get a free key at: [aistudio.google.com](https://aistudio.google.com/app/apikey)

---

### VS Code / Cursor Setup

**VS Code with GitHub Copilot:**
- Install extension: `GitHub.copilot` and `GitHub.copilot-chat`
- Sign in with GitHub account at [github.com/settings/copilot](https://github.com/settings/copilot)
- Key bindings: `Cmd+I` (inline edit), `Cmd+Shift+I` (chat sidebar)

**Cursor (recommended for this lab):**
- Download from [cursor.sh](https://cursor.sh) — free tier available
- `Cmd+K` opens inline edit, `Cmd+L` opens chat, `@codebase` enables full project context
- Import VS Code settings: `Cursor > Import VS Code Settings`

---

### Common Errors & Fixes

**`ModuleNotFoundError: No module named 'google.generativeai'`**
> Run the first code cell (auto-installs) then **restart the kernel** and re-run.

**`ModuleNotFoundError: No module named '...'`**
> Run the first cell or: `pip install <package>` in a terminal, then restart the kernel.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> Using a system Python. Open terminal, `source myenv/bin/activate`, then `pip install <package>` without that flag.

**Packages install but `ModuleNotFoundError` still appears**
> Wrong Python environment. Check the kernel shown top-right in VS Code and install into that env.

**`PermissionError` or `[Errno 13]` when installing**
> Use a virtual environment: `python -m venv myenv && source myenv/bin/activate` then install.

**Python version error**
> This lab requires Python 3.9+. Check: `python --version`. Upgrade if needed.

In [1]:
import subprocess, sys

required = {
    'google.generativeai': 'google-generativeai',
    'pandas':              'pandas',
}
for pkg, inst in required.items():
    try:
        __import__(pkg)
        print(f'  {pkg} already installed')
    except ImportError:
        print(f'  Installing {inst}...')
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', inst, '--quiet', '--break-system-packages']
        )
        print(f'  {inst} installed')

print('\nAll packages ready')

/var/folders/nr/zr3cbb4j2dz42fkzm9zw80080000gp/T/ipykernel_20357/3480270164.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  __import__(pkg)


  google.generativeai already installed
  pandas already installed

All packages ready


In [3]:
import os
import google.generativeai as genai

# Set your API key here (workshop only — use env vars in production)
# os.environ['GEMINI_API_KEY'] = 'AIza...'   # <- uncomment and paste key

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    # Try loading from .env file
    env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith('GEMINI_API_KEY='):
                    GEMINI_API_KEY = line.split('=', 1)[1].strip()
                    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
                    print('Loaded GEMINI_API_KEY from .env file')
                    break

if not GEMINI_API_KEY:
    raise EnvironmentError(
        'GEMINI_API_KEY not set.\n'
        'Option 1: Uncomment the os.environ line above and paste your key.\n'
        'Option 2: export GEMINI_API_KEY="AIza..." in your terminal.\n'
        'Option 3: Add GEMINI_API_KEY=AIza... to a .env file in the project root.\n'
        'Get a free key at: https://aistudio.google.com/app/apikey'
    )

genai.configure(api_key=GEMINI_API_KEY)
MODEL = 'gemini-3.1-flash-lite'   # fast and cost-effective for workshop demos

# ── Shared CALL_LOG tracks all API usage across this lab ───────────────────
CALL_LOG = []

def call_gemini(
    messages,
    system: str = None,
    max_tokens: int = 1024,
    temperature: float = 0.2,
    label: str = 'call',
) -> str:
    """Call Gemini and append token usage to CALL_LOG.

    Args:
        messages: List of dicts with 'role' and 'content' keys.
        system: Optional system instruction string.
        max_tokens: Maximum output tokens to generate.
        temperature: Sampling temperature (0=deterministic, 1=creative).
        label: Tag for cost tracking in CALL_LOG.

    Returns:
        Generated text as a string.
    """
    model_inst = genai.GenerativeModel(
        model_name=MODEL,
        system_instruction=system,
    )
    user_content = messages[-1]['content']
    response = model_inst.generate_content(
        user_content,
        generation_config=genai.GenerationConfig(
            max_output_tokens=max_tokens,
            temperature=temperature,
        )
    )
    in_tok  = response.usage_metadata.prompt_token_count
    out_tok = response.usage_metadata.candidates_token_count
    CALL_LOG.append({
        'label':         label,
        'input_tokens':  in_tok,
        'output_tokens': out_tok,
        'cost_usd':      (in_tok * 0.075 + out_tok * 0.30) / 1_000_000,
    })
    return response.text

# Quick smoke test
test_resp = call_gemini(
    messages=[{'role': 'user', 'content': 'Reply with exactly: ready'}],
    max_tokens=10,
    temperature=0.0,
    label='smoke_test'
)
print(f'Gemini API ready  |  model: {MODEL}')
print(f'Smoke test response: {test_resp.strip()}')

Gemini API ready  |  model: gemini-3.1-flash-lite
Smoke test response: ready


## Section 1 - The AI-Assisted Development Workflow

AI editor tools (Cursor, GitHub Copilot, VS Code Intellicode) send your code and instructions to an LLM backend — the same Gemini/GPT-4/Claude models you have been calling directly. Understanding this lets you use editor AI deliberately rather than treating it as a black box.

### Four Core Use Cases

| Use Case | What You Provide | What AI Returns | When to Use |
|----------|-----------------|-----------------|-------------|
| **Code Generation** | Natural-language description + context | Working code skeleton or full implementation | New classes, utility functions, boilerplate |
| **Refactoring** | Messy/legacy code | Clean, idiomatic, well-named version | Tech debt sprints, onboarding new engineers |
| **Documentation** | Undocumented code | Docstrings, inline comments, README | Pre-PR checklist, open-source releases |
| **Unit Test Generation** | Function signature + examples | pytest / unittest test cases | TDD, coverage gaps, regression suite |

### Manual vs AI-Assisted Workflow

| Step | Manual Approach | AI-Assisted Approach | Time Saved |
|------|----------------|---------------------|------------|
| Write new class | 20–40 min | Describe intent, review output | 60–70% |
| Rename variables + restructure | 15–30 min | Paste code, ask to refactor | 70–80% |
| Write docstrings | 10–20 min per file | Auto-generate, review | 80–90% |
| Write unit tests | 30–60 min | Generate from signature | 50–70% |
| Understand unfamiliar code | 20–60 min | Ask AI to explain | 60–80% |

> **Instructor Note:** Emphasise the review step. AI-generated code is a **first draft**, not production-ready output. The workflow is: generate fast, review carefully, commit deliberately. For Nutanix engineers this is especially important — log parsing and alert routing code paths are critical and must handle edge cases that the AI may miss (e.g., malformed log lines, unexpected severity levels, encoding issues).

## Section 2 - Code Generation

### The Pattern

When using AI to generate code, the quality of the output is directly proportional to the quality of your description. Provide:
1. **Role context** — what kind of engineer is writing this?
2. **Technology constraints** — language version, libraries allowed, style guide
3. **Exact interface** — class name, method names, parameter types, return types
4. **Example data** — show what inputs look like
5. **Error handling requirements** — what should fail gracefully vs raise?

In VS Code/Cursor this is the `Cmd+K` inline prompt or the chat sidebar. Here we call Gemini directly to see exactly what the tool sends and receives.

In [4]:
import pandas as pd

# ── System prompt: defines the AI's role and coding standards ──────────────
GENERATION_SYSTEM = """You are a senior Nutanix platform engineer writing production Python code.
Coding standards:
- Python 3.9+, no third-party imports (stdlib only)
- Type hints on all public methods
- Return empty list/dict rather than raising on missing data
- Use dataclasses or plain dicts for structured results
- Log format assumption: 'SEVERITY component module:line] message  key=val'
Write only the class code — no explanation, no markdown fences, no usage example."""

# ── Generation prompt: precise interface specification ─────────────────────
GENERATION_PROMPT = """Generate a Python class called NutanixLogParser with these methods:

1. parse_line(line: str) -> dict
   Parse a single AOS log line into keys: severity, component, location, message, kv_pairs (dict).
   Example input: 'FATAL stargate disk_manager.cc:412] disk_id=sda3 io_error=EIO retry_count=3'
   Expected output: {'severity': 'FATAL', 'component': 'stargate',
                     'location': 'disk_manager.cc:412', 'message': 'disk_id=sda3 io_error=EIO retry_count=3',
                     'kv_pairs': {'disk_id': 'sda3', 'io_error': 'EIO', 'retry_count': '3'}}
   Return empty dict with 'raw' key if line cannot be parsed.

2. parse_file(lines: list[str]) -> list[dict]
   Call parse_line on each line, skip empty lines, return list of parsed dicts.

3. filter_by_severity(records: list[dict], severities: list[str]) -> list[dict]
   Return only records whose severity is in the provided list (case-insensitive).

4. get_component_stats(records: list[dict]) -> dict
   Return dict mapping component name -> {'total': int, 'by_severity': dict}.
   Example: {'stargate': {'total': 3, 'by_severity': {'FATAL': 1, 'ERROR': 2}}}

The class should have no __init__ parameters."""

print('Sending generation prompt to Gemini...')
print(f'Prompt length: ~{len(GENERATION_PROMPT.split())} words')
print()

generated_code = call_gemini(
    messages=[{'role': 'user', 'content': GENERATION_PROMPT}],
    system=GENERATION_SYSTEM,
    max_tokens=1500,
    temperature=0.1,   # low temperature: deterministic, correct code
    label='code_generation'
)

# Strip markdown fences if the model added them despite instructions
if generated_code.startswith('```'):
    lines = generated_code.split('\n')
    generated_code = '\n'.join(lines[1:-1] if lines[-1] == '```' else lines[1:])
    generated_code = generated_code.lstrip('python\n')

print('=== Generated NutanixLogParser class ===')
print(generated_code)
print()

Sending generation prompt to Gemini...
Prompt length: ~133 words

=== Generated NutanixLogParser class ===
import typing
from dataclasses import dataclass

class NutanixLogParser:
    def parse_line(self, line: str) -> dict:
        line = line.strip()
        if not line:
            return {}

        parts = line.split(' ', 3)
        if len(parts) < 4:
            return {'raw': line}

        severity, component, location, message = parts
        
        kv_pairs = {}
        kv_tokens = message.split(' ')
        for token in kv_tokens:
            if '=' in token:
                key, _, val = token.partition('=')
                kv_pairs[key] = val

        return {
            'severity': severity,
            'component': component,
            'location': location,
            'message': message,
            'kv_pairs': kv_pairs
        }

    def parse_file(self, lines: list[str]) -> list[dict]:
        results = []
        for line in lines:
            if line.strip():
 

In [5]:
# ── exec() the generated class into the local namespace ───────────────────
exec(generated_code, globals())
print('NutanixLogParser class loaded into namespace')

# ── Test with realistic Nutanix AOS log lines ──────────────────────────────
SAMPLE_LOGS = [
    'FATAL stargate disk_manager.cc:412] disk_id=sda3 io_error=EIO retry_count=3 marking_disk_bad',
    'ERROR cerebro replication.cc:208] remote_site=DR-cluster rpo_violation=true lag_seconds=4312',
    'ERROR cassandra storage_engine.cc:991] keyspace=metadata compaction_failed=true error=ENOMEM',
    'WARNING stargate oplog.cc:87] oplog_full=true usage_pct=94 throttling_writes=true',
    'INFO curator disk_balancer.cc:156] rebalance_start=true migrating_gb=128 source_node=node-3',
]

parser = NutanixLogParser()
records = parser.parse_file(SAMPLE_LOGS)

print(f'Parsed {len(records)} log lines\n')

# Display as DataFrame
df = pd.DataFrame(records)
if 'kv_pairs' in df.columns:
    df = df.drop(columns=['kv_pairs'])  # exclude nested dict for display
pd.set_option('display.max_colwidth', 45)
print('=== Parsed Records ===')
print(df.to_string(index=False))
print()

# Test filter_by_severity
critical = parser.filter_by_severity(records, ['FATAL', 'ERROR'])
print(f'FATAL/ERROR records: {len(critical)}')
for r in critical:
    print(f"  [{r.get('severity')}] {r.get('component')} - {r.get('message', '')[:60]}")
print()

# Test get_component_stats
stats = parser.get_component_stats(records)
print('=== Component Stats ===')
for component, data in stats.items():
    print(f'  {component}: total={data["total"]}, by_severity={data["by_severity"]}')

NutanixLogParser class loaded into namespace
Parsed 5 log lines

=== Parsed Records ===
severity component               location                                                    message
   FATAL  stargate   disk_manager.cc:412]   disk_id=sda3 io_error=EIO retry_count=3 marking_disk_bad
   ERROR   cerebro    replication.cc:208] remote_site=DR-cluster rpo_violation=true lag_seconds=4312
   ERROR cassandra storage_engine.cc:991]      keyspace=metadata compaction_failed=true error=ENOMEM
 WARNING  stargate           oplog.cc:87]        oplog_full=true usage_pct=94 throttling_writes=true
    INFO   curator  disk_balancer.cc:156]   rebalance_start=true migrating_gb=128 source_node=node-3

FATAL/ERROR records: 3
  [FATAL] stargate - disk_id=sda3 io_error=EIO retry_count=3 marking_disk_bad
  [ERROR] cerebro - remote_site=DR-cluster rpo_violation=true lag_seconds=4312
  [ERROR] cassandra - keyspace=metadata compaction_failed=true error=ENOMEM

=== Component Stats ===
  stargate: total=2, by_

## Section 3 - Code Refactoring

### AI Refactoring: What It Does Well

Refactoring with AI is one of the highest-ROI use cases in developer tooling. Given messy legacy code, the model can:
- Rename cryptic variables (`x`, `d`, `temp`) to descriptive names
- Extract magic numbers into named constants
- Add error handling and input validation
- Restructure nested conditionals into guard clauses
- Add type hints throughout
- Remove dead code and consolidate duplication

In Cursor: select the messy function, press `Cmd+K`, type "refactor this to be clean and production-ready".

Here we do the same call explicitly so you can see the before/after diff and understand what the prompt sends.

In [6]:
# ── BEFORE: messy legacy function with all the common anti-patterns ────────
BEFORE_CODE = '''
def process_nutanix_alerts(d):
    x = []
    for i in d:
        temp = i['sev']
        if temp == 'FATAL' or temp == 'ERROR' or temp == 'CRITICAL':
            n = i.get('node')
            t = i.get('ts')
            msg = i.get('msg')
            if n != None:
                if t != None:
                    if msg != None:
                        r = {}
                        r['node'] = n
                        r['time'] = t
                        r['message'] = msg
                        r['sev'] = temp
                        if temp == 'FATAL':
                            r['p'] = 1
                        elif temp == 'CRITICAL':
                            r['p'] = 1
                        else:
                            r['p'] = 2
                        x.append(r)
    return x
'''

REFACTOR_SYSTEM = """You are a senior Python engineer performing a code review and refactoring.
Apply these improvements:
1. Add a descriptive docstring (Google style)
2. Add type hints to parameters and return value
3. Replace single-letter variable names with descriptive names
4. Extract magic numbers/strings into named constants at the top of the function
5. Replace nested if-chains with guard clauses (early return / continue)
6. Combine redundant conditions
7. Use dict literal syntax instead of assigning keys one by one
Return ONLY the refactored function — no explanation, no markdown fences."""

REFACTOR_PROMPT = f"""Refactor this Python function:

{BEFORE_CODE}"""

print('Sending refactoring request to Gemini...')
refactored_code = call_gemini(
    messages=[{'role': 'user', 'content': REFACTOR_PROMPT}],
    system=REFACTOR_SYSTEM,
    max_tokens=800,
    temperature=0.1,
    label='refactoring'
)

# Strip markdown fences if present
if refactored_code.strip().startswith('```'):
    lines = refactored_code.strip().split('\n')
    refactored_code = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    refactored_code = refactored_code.lstrip('python\n')

print('=' * 55)
print('BEFORE (messy legacy code):')
print('=' * 55)
print(BEFORE_CODE)
print('=' * 55)
print('AFTER (AI-refactored):')
print('=' * 55)
print(refactored_code)

Sending refactoring request to Gemini...
BEFORE (messy legacy code):

def process_nutanix_alerts(d):
    x = []
    for i in d:
        temp = i['sev']
        if temp == 'FATAL' or temp == 'ERROR' or temp == 'CRITICAL':
            n = i.get('node')
            t = i.get('ts')
            msg = i.get('msg')
            if n != None:
                if t != None:
                    if msg != None:
                        r = {}
                        r['node'] = n
                        r['time'] = t
                        r['message'] = msg
                        r['sev'] = temp
                        if temp == 'FATAL':
                            r['p'] = 1
                        elif temp == 'CRITICAL':
                            r['p'] = 1
                        else:
                            r['p'] = 2
                        x.append(r)
    return x

AFTER (AI-refactored):
PRIORITY_HIGH = 1
PRIORITY_MEDIUM = 2
HIGH_SEVERITY_LEVELS = {'FATAL', 'ERROR', 'CRITICAL'}
FAT

In [7]:
# ── Confirm the refactored function actually runs ──────────────────────────
exec(refactored_code, globals())

# Find the function name (it may have been renamed during refactoring)
import inspect
func_name = None
for line in refactored_code.split('\n'):
    if line.strip().startswith('def '):
        func_name = line.strip().split('(')[0].replace('def ', '')
        break

if func_name:
    print(f'Refactored function name: {func_name}')
    # Test with sample alert data
    sample_alerts = [
        {'sev': 'FATAL',    'node': 'node-2', 'ts': 1706025600, 'msg': 'disk_io_error on sda3'},
        {'sev': 'WARNING',  'node': 'node-1', 'ts': 1706025700, 'msg': 'high_memory_usage'},
        {'sev': 'ERROR',    'node': 'node-3', 'ts': 1706025800, 'msg': 'replication_lag exceeded'},
        {'sev': 'INFO',     'node': 'node-1', 'ts': 1706025900, 'msg': 'backup completed'},
        {'sev': 'CRITICAL', 'node': 'node-4', 'ts': 1706026000, 'msg': 'cvm_unreachable'},
        {'sev': 'FATAL',    'node': None,     'ts': 1706026100, 'msg': 'oom_kill'},  # missing node
    ]
    func = globals()[func_name]
    result = func(sample_alerts)
    print(f'Input:  {len(sample_alerts)} alerts')
    print(f'Output: {len(result)} critical/error alerts (excluded INFO, WARNING, missing fields)')
    print()
    for r in result:
        print(f'  {r}')
else:
    print('Could not detect function name — check refactored_code above')

Refactored function name: process_nutanix_alerts
Input:  6 alerts
Output: 3 critical/error alerts (excluded INFO, WARNING, missing fields)

  {'node': 'node-2', 'time': 1706025600, 'message': 'disk_io_error on sda3', 'sev': 'FATAL', 'p': 1}
  {'node': 'node-3', 'time': 1706025800, 'message': 'replication_lag exceeded', 'sev': 'ERROR', 'p': 2}
  {'node': 'node-4', 'time': 1706026000, 'message': 'cvm_unreachable', 'sev': 'CRITICAL', 'p': 1}


## Section 4 - AI Documentation

### Auto-Generating Docstrings and Documentation

Documentation is often the last thing engineers write and the first thing that becomes outdated. AI can:
- Add **Google-style** or **NumPy-style** docstrings to every method
- Generate **inline comments** explaining non-obvious logic
- Write a **README.md** from a codebase description
- Produce **API reference** from type signatures

In Cursor: right-click a function → "Add Docstring", or select a file and chat: "add Google-style docstrings to all public methods".

The key insight: documentation prompts work best at **temperature 0.0–0.2** (accurate, not creative) and with a **specific style guide** in the system prompt.

In [8]:
# ── Ask Gemini to add Google-style docstrings to NutanixLogParser ──────────
DOC_SYSTEM = """You are a technical writer adding documentation to Python code.
Rules:
- Add Google-style docstrings to the class and every public method
- Include Args, Returns, Raises, and Example sections where appropriate
- Add inline comments on lines with non-obvious logic
- Do NOT change any logic — only add documentation
- Return ONLY the documented Python code, no markdown fences, no explanation"""

DOC_PROMPT = f"""Add comprehensive Google-style docstrings and inline comments to this class:

{generated_code}"""

print('Generating documentation for NutanixLogParser...')
documented_code = call_gemini(
    messages=[{'role': 'user', 'content': DOC_PROMPT}],
    system=DOC_SYSTEM,
    max_tokens=2000,
    temperature=0.0,   # documentation must be precise, not creative
    label='docstring_generation'
)

# Strip fences if present
if documented_code.strip().startswith('```'):
    lines = documented_code.strip().split('\n')
    documented_code = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    documented_code = documented_code.lstrip('python\n')

print('=== NutanixLogParser with Google-style docstrings ===')
print(documented_code)

Generating documentation for NutanixLogParser...
=== NutanixLogParser with Google-style docstrings ===
import typing
from dataclasses import dataclass

class NutanixLogParser:
    """A parser for Nutanix-formatted log lines and collections of log records."""

    def parse_line(self, line: str) -> dict:
        """Parses a single log line into a structured dictionary.

        Args:
            line: A string representing a single line from a Nutanix log file.

        Returns:
            A dictionary containing parsed fields (severity, component, location, 
            message, kv_pairs) or a dictionary with the raw line if parsing fails.
            Returns an empty dictionary if the input line is empty.
        """
        line = line.strip()
        if not line:
            return {}

        # Split into max 4 parts: severity, component, location, and the rest as message
        parts = line.split(' ', 3)
        if len(parts) < 4:
            return {'raw': line}

        severi

In [9]:
# ── Generate a README.md for the Nutanix AIOps monitoring script ───────────
README_SYSTEM = """You are a developer advocate writing clear, concise README files.
Format: GitHub-flavoured Markdown.
Sections required: title, one-line description, Features list, Requirements,
Installation, Usage (with code example), Configuration, Contributing, License.
Tone: professional, direct. No fluff. Target audience: SREs and platform engineers."""

README_PROMPT = """Write a README.md for a Python script called nutanix_aiops_monitor.py.

Description: Monitors a Nutanix AOS cluster by tailing CVM log files in real time,
parsing log lines with NutanixLogParser, detecting FATAL/ERROR events, and sending
alerts to a Slack webhook with component name, node, severity, and a suggested
remediation action from a Gemini API call.

Configuration: GEMINI_API_KEY env var, SLACK_WEBHOOK_URL env var,
LOG_PATH (default /home/nutanix/data/logs/stargate.out),
SEVERITY_FILTER (comma-separated, default FATAL,ERROR)."""

print('Generating README.md...')
readme_content = call_gemini(
    messages=[{'role': 'user', 'content': README_PROMPT}],
    system=README_SYSTEM,
    max_tokens=1200,
    temperature=0.2,
    label='readme_generation'
)

print('=== Generated README.md ===')
print(readme_content)

Generating README.md...
=== Generated README.md ===
# nutanix_aiops_monitor.py

A real-time monitoring agent for Nutanix AOS clusters that parses CVM logs, detects critical events, and provides AI-driven remediation suggestions via Slack.

## Features
* **Real-time Tailing:** Monitors CVM log files for new entries as they are written.
* **Intelligent Parsing:** Utilizes `NutanixLogParser` to structure raw log data.
* **Automated Diagnostics:** Integrates with Google Gemini API to generate remediation steps for detected issues.
* **Slack Integration:** Delivers formatted alerts including component, node, severity, and remediation advice.
* **Configurable Filtering:** Define specific severity levels to monitor.

## Requirements
* Python 3.8+
* Access to Nutanix CVM (Controller VM)
* `NutanixLogParser` library
* Google Gemini API Key
* Slack Incoming Webhook URL

## Installation
1. Clone the repository to your CVM or management node:
   ```bash
   git clone <repository-url>
   cd nutanix-

## Section 5 - Unit Test Generation

### AI-Generated Unit Tests

Unit test generation is where AI-assisted development gives some of the clearest ROI:
- Engineers spend 30–60 minutes writing thorough test suites manually
- AI can generate a comprehensive first draft in seconds
- The engineer's job shifts from **writing** tests to **reviewing** them for edge cases

The most effective prompt pattern:
1. Provide the **exact function signature and docstring**
2. Describe **edge cases** you care about (empty input, malformed data, boundary values)
3. Specify the **test framework** (pytest, unittest)
4. Ask for **both positive and negative tests**

In Cursor: select a function, `Cmd+L` (chat), type: "generate pytest tests covering all edge cases including malformed input".

In [10]:
# ── Target function to test ────────────────────────────────────────────────
def parse_nutanix_alert(s: str) -> dict:
    """Parse a pipe-delimited Nutanix alert string into a structured dict.

    Args:
        s: Alert string in format 'SEVERITY|component|event_type|node|timestamp'.
           Example: 'CRITICAL|stargate|disk_io_error|node-2|1706025600'

    Returns:
        Dict with keys: severity, component, event_type, node, timestamp (int).
        Returns empty dict if input is malformed.
    """
    if not s or not isinstance(s, str):
        return {}
    parts = s.strip().split('|')
    if len(parts) != 5:
        return {}
    severity, component, event_type, node, ts_str = parts
    try:
        timestamp = int(ts_str)
    except ValueError:
        return {}
    return {
        'severity':   severity.upper(),
        'component':  component,
        'event_type': event_type,
        'node':       node,
        'timestamp':  timestamp,
    }

# Verify it works before asking AI to test it
test_input = 'CRITICAL|stargate|disk_io_error|node-2|1706025600'
print('parse_nutanix_alert result:')
print(parse_nutanix_alert(test_input))
print()
print('Malformed input result:', parse_nutanix_alert('bad|data'))
print('Empty input result:', parse_nutanix_alert(''))

parse_nutanix_alert result:
{'severity': 'CRITICAL', 'component': 'stargate', 'event_type': 'disk_io_error', 'node': 'node-2', 'timestamp': 1706025600}

Malformed input result: {}
Empty input result: {}


In [11]:
import inspect

TEST_SYSTEM = """You are a senior Python QA engineer writing pytest unit tests.
Rules:
- Use unittest.TestCase (not bare pytest functions) so tests can run without pytest installed
- Class name: TestParseNutanixAlert
- Write at least 8 test methods covering: valid input, empty string, None, wrong field count,
  non-integer timestamp, extra whitespace, different severity levels, uppercase normalisation
- Each test method has a single assertion and a clear name like test_valid_returns_severity
- Import only from stdlib (unittest, the function under test is already in scope)
- Do NOT include any import for the function itself — it will be injected before the tests run
- Return ONLY the test class code, no markdown fences, no explanation"""

func_source = inspect.getsource(parse_nutanix_alert)

TEST_PROMPT = f"""Generate unittest.TestCase tests for this function:

{func_source}

The function parse_nutanix_alert will be available in scope when tests run.
Do not import it."""

print('Generating unit tests...')
generated_tests = call_gemini(
    messages=[{'role': 'user', 'content': TEST_PROMPT}],
    system=TEST_SYSTEM,
    max_tokens=1500,
    temperature=0.1,
    label='test_generation'
)

# Strip markdown fences if present
if generated_tests.strip().startswith('```'):
    lines = generated_tests.strip().split('\n')
    generated_tests = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    generated_tests = generated_tests.lstrip('python\n')

print('=== Generated Tests ===')
print(generated_tests)

Generating unit tests...
=== Generated Tests ===
class TestParseNutanixAlert(unittest.TestCase):

    def test_valid_returns_correct_dict(self):
        self.assertEqual(parse_nutanix_alert('CRITICAL|stargate|disk_io_error|node-2|1706025600'), {'severity': 'CRITICAL', 'component': 'stargate', 'event_type': 'disk_io_error', 'node': 'node-2', 'timestamp': 1706025600})

    def test_empty_string_returns_empty_dict(self):
        self.assertEqual(parse_nutanix_alert(''), {})

    def test_none_input_returns_empty_dict(self):
        self.assertEqual(parse_nutanix_alert(None), {})

    def test_wrong_field_count_returns_empty_dict(self):
        self.assertEqual(parse_nutanix_alert('CRITICAL|stargate|node-2|1706025600'), {})

    def test_non_integer_timestamp_returns_empty_dict(self):
        self.assertEqual(parse_nutanix_alert('CRITICAL|stargate|disk_io_error|node-2|abc'), {})

    def test_extra_whitespace_is_stripped(self):
        self.assertEqual(parse_nutanix_alert('  INFO|cluster|t

In [12]:
import unittest

# ── Inject the function into the test namespace, then exec the test class ──
exec_ns = {'parse_nutanix_alert': parse_nutanix_alert, 'unittest': unittest}
exec(generated_tests, exec_ns)

# Find the test class in the executed namespace
test_class = None
for name, obj in exec_ns.items():
    try:
        if isinstance(obj, type) and issubclass(obj, unittest.TestCase) and obj is not unittest.TestCase:
            test_class = obj
            print(f'Found test class: {name}')
            break
    except TypeError:
        continue

if test_class is None:
    print('ERROR: No unittest.TestCase subclass found in generated tests.')
    print('Review generated_tests output above.')
else:
    # Run the test suite
    suite  = unittest.TestLoader().loadTestsFromTestCase(test_class)
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)

    print()
    print('=' * 50)
    total  = result.testsRun
    passed = total - len(result.failures) - len(result.errors)
    print(f'RESULTS: {passed}/{total} passed')
    if result.failures:
        print(f'Failures ({len(result.failures)}):')
        for test, traceback in result.failures:
            print(f'  FAIL: {test}')
    if result.errors:
        print(f'Errors ({len(result.errors)}):')
        for test, traceback in result.errors:
            print(f'  ERROR: {test}')

test_different_severity_levels_are_handled (builtins.TestParseNutanixAlert.test_different_severity_levels_are_handled) ... ok
test_empty_string_returns_empty_dict (builtins.TestParseNutanixAlert.test_empty_string_returns_empty_dict) ... ok
test_extra_whitespace_is_stripped (builtins.TestParseNutanixAlert.test_extra_whitespace_is_stripped) ... ok
test_non_integer_timestamp_returns_empty_dict (builtins.TestParseNutanixAlert.test_non_integer_timestamp_returns_empty_dict) ... ok
test_none_input_returns_empty_dict (builtins.TestParseNutanixAlert.test_none_input_returns_empty_dict) ... ok
test_severity_is_normalised_to_uppercase (builtins.TestParseNutanixAlert.test_severity_is_normalised_to_uppercase) ... ok
test_valid_returns_correct_dict (builtins.TestParseNutanixAlert.test_valid_returns_correct_dict) ... ok
test_wrong_field_count_returns_empty_dict (builtins.TestParseNutanixAlert.test_wrong_field_count_returns_empty_dict) ... ok

-----------------------------------------------------------

Found test class: TestParseNutanixAlert

RESULTS: 8/8 passed


## Section 6 - VS Code / Cursor Reference

### Keyboard Shortcuts

| Action | VS Code + Copilot | Cursor |
|--------|------------------|--------|
| Inline edit (generate/replace selection) | `Cmd+I` | `Cmd+K` |
| Open AI chat sidebar | `Cmd+Shift+I` | `Cmd+L` |
| Accept inline suggestion | `Tab` | `Tab` |
| Reject inline suggestion | `Esc` | `Esc` |
| Explain selected code | Right-click → Copilot → Explain | `Cmd+L` → "explain this" |
| Fix error / quick fix | `Cmd+.` → Copilot Fix | `Cmd+K` → "fix this error" |
| Generate tests for function | Right-click → Copilot → Generate Tests | Select fn → `Cmd+L` → "write tests" |
| Add docstring | Right-click → Copilot → Generate Docs | Select fn → `Cmd+K` → "add docstring" |

### How `@codebase` Works in Cursor

When you type `@codebase` in Cursor's chat, it:
1. **Indexes** your project files into a vector store (runs locally on first open)
2. **Embeds** your query and retrieves the most relevant code chunks (same as RAG from Lab 5.4)
3. **Prepends** those chunks as context before your message in the system prompt
4. **Sends** the full prompt (context + query) to the configured LLM backend

This is exactly what Section 7 of this lab replicates manually.

### Cursor AI Model Selection

| Model | Best For | Context Window |
|-------|----------|---------------|
| GPT-4o | Complex refactoring, architecture | 128k tokens |
| Claude Sonnet 4 | Long file analysis, documentation | 200k tokens |
| Gemini 1.5 Flash | Fast generation, test writing | 1M tokens |
| cursor-small | Autocomplete, simple edits | 32k tokens |

> **Instructor Note:** Have attendees open Cursor and try `Cmd+K` on a function from the NutanixLogParser. The experience of the notebook (calling Gemini explicitly) and the editor (inline AI) should feel identical now that they understand what is happening under the hood. Key point: the editor is not smarter than the API — it is the same API with a better UI and automatic context injection.

In [13]:
# ── Section 7: Codebase Context Simulation ────────────────────────────────
# This replicates exactly what Cursor @codebase does:
# 1. Read relevant code from the project
# 2. Prepend it as context in the system prompt
# 3. Ask a question about extending or modifying it

print('=== Simulating Cursor @codebase Context Injection ===')
print()

# Step 1: "Index" — in a real tool this would be a vector search over all files.
# Here we manually select the most relevant snippet (NutanixLogParser).
CODEBASE_CONTEXT = generated_code  # the class we generated in Section 2

print(f'Context snippet: NutanixLogParser ({len(CODEBASE_CONTEXT.split(chr(10)))} lines)')
print()

# Step 2: Build a system prompt that includes the codebase context
CODEBASE_SYSTEM = f"""You are an expert Python engineer working on the Nutanix AIOps monitoring project.
Here is the current codebase context:

--- FILE: nutanix_log_parser.py ---
{CODEBASE_CONTEXT}
--- END FILE ---

Answer questions about this code and generate extensions that are consistent with its style.
Return only code when asked to add methods or classes."""

# Step 3: Ask a question that requires understanding the existing code
CODEBASE_QUERY = """Add a new method to NutanixLogParser called get_timeline(records: list[dict], 
window_seconds: int = 60) -> list[dict] that groups parsed log records into time buckets 
of window_seconds duration and returns a list of dicts with keys: 
bucket_start (int), bucket_end (int), event_count (int), severity_counts (dict), components (list).
Only write the new method — not the entire class."""

print(f'User query: "{CODEBASE_QUERY[:80]}..."')
print()
print('Sending to Gemini with codebase context prepended...')
print()

codebase_response = call_gemini(
    messages=[{'role': 'user', 'content': CODEBASE_QUERY}],
    system=CODEBASE_SYSTEM,
    max_tokens=800,
    temperature=0.1,
    label='codebase_context'
)

# Strip fences
if codebase_response.strip().startswith('```'):
    lines = codebase_response.strip().split('\n')
    codebase_response = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    codebase_response = codebase_response.lstrip('python\n')

print('=== Gemini response (context-aware extension) ===')
print(codebase_response)
print()
print('--- What @codebase did ---')
print(f'1. Injected {len(CODEBASE_CONTEXT)} chars of project code as system context')
print(f'2. Sent user query ({len(CODEBASE_QUERY)} chars)')
print(f'3. Model used existing method names/style from the injected context')
print('4. Result is a drop-in extension consistent with the existing class')

=== Simulating Cursor @codebase Context Injection ===

Context snippet: NutanixLogParser (63 lines)

User query: "Add a new method to NutanixLogParser called get_timeline(records: list[dict], 
w..."

Sending to Gemini with codebase context prepended...

=== Gemini response (context-aware extension) ===
    def get_timeline(self, records: list[dict], window_seconds: int = 60) -> list[dict]:
        if not records:
            return []

        # Assuming records contain a 'timestamp' field in epoch seconds.
        # If not present, we filter out records without timestamps.
        timed_records = [r for r in records if 'timestamp' in r]
        if not timed_records:
            return []

        timed_records.sort(key=lambda x: x['timestamp'])
        start_time = timed_records[0]['timestamp']
        
        buckets = {}
        for record in timed_records:
            ts = record['timestamp']
            bucket_idx = (ts - start_time) // window_seconds
            bucket_start = s

## Summary

This lab covered the four AI-assisted development workflows and their direct mapping to editor tools:

| Technique | When to Use | Nutanix Example | Editor Shortcut |
|-----------|-------------|-----------------|----------------|
| **Code Generation** | New class, utility, boilerplate from description | Generate NutanixLogParser from interface spec | `Cmd+K` in Cursor |
| **Refactoring** | Legacy code, tech debt, pre-PR cleanup | Clean up process_nutanix_alerts() | Select + `Cmd+K` "refactor" |
| **Docstring Generation** | Pre-PR, open-source release, onboarding | Add Google docstrings to LogParser | Select class + `Cmd+K` "add docstrings" |
| **Unit Test Generation** | New functions, coverage gaps, regression | Generate pytest for parse_nutanix_alert | Select fn + `Cmd+L` "write tests" |
| **Codebase Context (@codebase)** | Extending existing code, understanding unfamiliar code | Add get_timeline() consistent with existing style | `@codebase` in Cursor chat |
| **README Generation** | New project, documentation sprint | nutanix_aiops_monitor.py README | Chat: "write a README for this project" |

### Key Takeaways

1. **AI editor tools are API calls** — understanding Gemini/GPT prompts makes you a better Cursor/Copilot user
2. **Temperature matters** — use 0.0–0.1 for code/docs (correct), 0.5–0.7 for brainstorming
3. **Review everything** — AI code is a fast first draft, not production-ready
4. **Context is king** — `@codebase` works because it injects relevant code before your question
5. **System prompts set standards** — a good system prompt encodes your team's style guide

### Next Steps

In **Lab 6.2** we extend this workflow to **CI/CD integration**: auto-generating PR descriptions, running AI-powered code review as a GitHub Action, and building a Slack bot that explains failing test output using the same Gemini API patterns from this lab.

## Challenges

### Challenge 1 — Context Window Budget

Large codebases exceed the context window. Implement a `prioritize_context(files: list[str], query: str, max_tokens: int = 4000) -> str` function that:
1. Scores each file by keyword overlap with the query
2. Greedily adds files (highest score first) until the token budget is exhausted
3. Returns the combined context string

Test it with at least 5 simulated code files and a query about log parsing.

### Challenge 2 — Test Coverage Completeness

The generated tests for `parse_nutanix_alert` may have missed some edge cases. Identify at least **3 additional test cases** that the AI did not generate (consider: Unicode characters in fields, extra pipe characters in message, very large timestamp values, severity with mixed case). Add them to the test class and confirm they pass.

### Challenge 3 — Refactoring Chain

Apply a 3-step refactoring chain:
1. First AI call: refactor `process_nutanix_alerts()` for clean code (done in Section 3)
2. Second AI call: take the refactored output and ask the AI to add type hints and a dataclass for the return value
3. Third AI call: take the typed output and generate unit tests for it

Compare the quality of tests generated after the 3-step chain vs tests generated from the original messy code. Which test suite is more comprehensive and why?

In [ ]:
# Challenge workspace — write your solutions here
